In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.titleweight'] = 'bold'

DATA = 'data'

def to_dt(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors='coerce'), unit='ms')

In [ ]:
labels = pd.read_csv(f'{DATA}/incident_label.csv')
labels['ts_start_dt'] = to_dt(labels['ts_start'])
labels['ts_end_dt'] = to_dt(labels['ts_end'])
labels['ts_end_f'] = labels['ts_end'].fillna(labels['ts_start'] + 3_600_000)
labels[['id', 'tier', 'kind', 'severity', 'provenance', 'ts_start_dt']].sort_values('ts_start_dt')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = {1: '#d62728', 2: '#ff7f0e', 3: '#7f7f7f'}
for tier, grp in labels.groupby('tier'):
    is_lt = grp['kind'] == 'load_test'
    ax.scatter(grp.loc[~is_lt, 'ts_start_dt'], [tier]*(~is_lt).sum(),
               color=colors[tier], s=80, label=f'tier {tier}', zorder=3)
    ax.scatter(grp.loc[is_lt, 'ts_start_dt'], [tier]*is_lt.sum(),
               color=colors[tier], s=80, marker='x', zorder=3)
ax.scatter([], [], color='gray', marker='x', s=80, label='load_test (não-incidente)')
ax.set_yticks([1, 2, 3]); ax.set_ylim(0.5, 3.5)
ax.set_ylabel('tier'); ax.set_title(f'Os {len(labels)} rótulos do golden set, no tempo (X = teste de carga, não incidente)')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b'))
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

print(f"Total: {len(labels)} linhas | {(labels['kind']!='load_test').sum()} incidentes reais | "
      f"{(labels['kind']=='load_test').sum()} load_test (excluir)")
print("\nDistribuição por provenance (confiabilidade do rótulo):")
print(labels['provenance'].value_counts())

In [ ]:
inc = pd.read_csv(f'{DATA}/incidents.csv')
inc['opened_dt'] = to_dt(inc['opened_at'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
inc['priority'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribuição de prioridade (783 chamados brutos)')
axes[0].set_xlabel('priority'); axes[0].set_ylabel('qtde')

video_mask = inc['business_service'].fillna('').str.contains('Globoplay|[Vv]ídeo', regex=True)
relevant = inc[inc['priority'].isin(['P1', 'P2', 'P3']) & video_mask]
inc['relevante'] = 'ruído (P4/P5 ou não-vídeo)'
inc.loc[inc['priority'].isin(['P1','P2','P3']) & video_mask, 'relevante'] = 'P1-P3 + vídeo/Globoplay'
inc['relevante'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.0f%%', ylabel='')
axes[1].set_title('Quanto do chamado bruto é sinal de verdade')
plt.tight_layout(); plt.show()

print(f"P1-P3 + vídeo/Globoplay: {len(relevant)} chamados")
already = set(labels['id'].str.replace('itsm-', '', regex=False))
novos = sorted(set(relevant['sys_id']) - set())  # sys_id != number; comparar por número
novos_num = sorted(set(relevant['number']) - already)
print(f"Já estão no golden set: {len(set(relevant['number']) & already)}")
print(f"Ainda não rotulados (dá pra adicionar de graça, olhando o ticket): {len(novos_num)} -> {novos_num}")

In [ ]:
sb = pd.read_csv(f'{DATA}/signal_bucket.csv', dtype={'source': 'category', 'entity': 'category', 'metric': 'category'})
sb['dt'] = to_dt(sb['bts'])

fig, ax = plt.subplots(figsize=(11, 4))
sb['source'].value_counts().plot(kind='barh', ax=ax, color='#55A868')
ax.set_title(f'Volume de pontos por fonte ({len(sb):,} pontos, {sb["entity"].nunique()} entidades, {sb["metric"].nunique()} métricas)')
ax.set_xlabel('pontos')
plt.tight_layout(); plt.show()

In [ ]:
stats = sb.groupby(['source', 'entity', 'metric'], observed=True)['value'].agg(['mean', 'std', 'count'])
stats = stats[(stats['std'] > 1e-9) & (stats['count'] >= 30)]
sb2 = sb.merge(stats, on=['source', 'entity', 'metric'], how='inner')
sb2['z'] = (sb2['value'] - sb2['mean']) / sb2['std']

anomalias = sb2[sb2['z'].abs() > 4].copy().sort_values('z', ascending=False)
print(f"{len(stats)} séries analisadas | {len(anomalias)} pontos com |z|>4 ({len(anomalias)/len(sb2)*100:.3f}% do total)")
anomalias[['dt', 'source', 'entity', 'metric', 'value', 'z']].head(15)

In [ ]:
# Plot da série com o maior z-score isolado (mostra o método funcionando em dado real)
top = anomalias.iloc[0]
serie = sb2[(sb2['source']==top['source']) & (sb2['entity']==top['entity']) & (sb2['metric']==top['metric'])].sort_values('dt')

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(serie['dt'], serie['value'], color='#4C72B0', lw=1, label='valor observado')
m, s = serie['mean'].iloc[0], serie['std'].iloc[0]
ax.axhline(m, color='gray', ls='--', lw=1, label='baseline (média)')
ax.fill_between(serie['dt'], m-4*s, m+4*s, color='gray', alpha=0.15, label='faixa normal (±4σ)')
flag = serie[serie['z'].abs() > 4]
ax.scatter(flag['dt'], flag['value'], color='#d62728', s=40, zorder=5, label='anomalia (|z|>4)')

for _, row in labels.iterrows():
    if pd.isna(row['ts_start_dt']):
        continue
    end = row['ts_end_dt'] if pd.notna(row['ts_end_dt']) else row['ts_start_dt'] + pd.Timedelta(hours=1)
    ax.axvspan(row['ts_start_dt'], end, color='orange' if row['kind']=='incident' else 'green', alpha=0.15)

ax.set_title(f"{top['source']} | {top['entity']} | {top['metric']}  (laranja = janela de incidente rotulado, verde = load_test)")
ax.legend(loc='upper left', fontsize=8)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
incidentes_reais = labels[labels['kind'] == 'incident'].copy()
incidentes_reais['ts_end_filled'] = incidentes_reais['ts_end'].fillna(incidentes_reais['ts_start'] + 3_600_000)
iidx = pd.IntervalIndex.from_arrays(incidentes_reais['ts_start'], incidentes_reais['ts_end_filled'], closed='both')

def bate_incidente(ts):
    try:
        return iidx.get_indexer([ts])[0] != -1
    except Exception:
        return False

amostra = anomalias.head(300).copy()  # amostra das maiores anomalias, checagem é O(log n) mas por segurança
amostra['dentro_de_incidente_conhecido'] = amostra['bts'].apply(bate_incidente)

resumo = amostra['dentro_de_incidente_conhecido'].value_counts()
print("Das 300 maiores anomalias:")
print(f"  dentro de janela já rotulada: {resumo.get(True, 0)}")
print(f"  FORA de qualquer janela rotulada (candidato novo): {resumo.get(False, 0)}")
print("\nExemplos de candidatos não rotulados (maior z primeiro):")
amostra[~amostra['dentro_de_incidente_conhecido']][['dt','source','entity','metric','value','z']].head(10)

In [ ]:
qoe = pd.read_csv(f'{DATA}/live_qoe_hourly.csv')
qoe['dt'] = to_dt(qoe['hour_ts'])
qoe_agg = qoe.groupby('dt', as_index=False)[['latency_ms_p95', 'bufferratio_avg', 'stalls_avg']].mean()

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(qoe_agg['dt'], qoe_agg['latency_ms_p95'], color='#4C72B0'); axes[0].set_title('Latência de início (p95, ms)')
axes[1].plot(qoe_agg['dt'], qoe_agg['bufferratio_avg'], color='#DD8452'); axes[1].set_title('Buffer ratio médio (fração do tempo travado)')
axes[2].plot(qoe_agg['dt'], qoe_agg['stalls_avg'], color='#C44E52'); axes[2].set_title('Travamentos médios por sessão')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()